# Pix2Pix MoNuSeg — Inference Demo

This notebook demonstrates inference with the trained Pix2Pix models on paired images from the MoNuSeg test set.

Each paired image contains:
- **Left half**: real H&E-stained histology image.
- **Right half**: corresponding label map (nuclei segmentation).

Our generator takes the label map as input and produces a realistic histology image. We then display the input label, the generated image, and the ground-truth real image side by side.

The notebook reuses the same inference functions as the command-line demo (`demo/demo.py`), so the results are identical.

## 1. Setup

We import the inference utilities from `inference.py`. Make sure you have run `pip install -r requirements.txt` from the project root before executing this notebook.

In [ ]:
import sys
from pathlib import Path

# Allow imports relative to the project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "demo") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "demo"))

import matplotlib.pyplot as plt

from inference import (
    AVAILABLE_MODELS,
    load_generator,
    resolve_checkpoint_path,
    run_inference_on_paired_image,
    make_side_by_side_figure,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Available models: {list(AVAILABLE_MODELS)}")

## 2. Choose a model and an input image

By default we use the **best improved model** (augmentation + LSGAN + multi-scale discriminator), but you can switch to any of the five trained variants by changing `MODEL_NAME` below.

The first time you run a given model, the checkpoint will be downloaded from Hugging Face Hub (~200 MB) and cached locally.

In [ ]:
MODEL_NAME = "improved"  # one of: baseline, aug, lsgan, multi, improved

# Pick the first sample image found in samples/
samples_dir = PROJECT_ROOT / "samples"
sample_images = sorted(samples_dir.glob("*.png"))

if not sample_images:
    raise FileNotFoundError(
        f"No sample images found in {samples_dir}. "
        "Please add some test-set images there (see samples/README.md)."
    )

INPUT_IMAGE = sample_images[0]
print(f"Input image: {INPUT_IMAGE.name}")
print(f"Model: {MODEL_NAME}")

## 3. Load the generator

In [ ]:
checkpoint_path = resolve_checkpoint_path(MODEL_NAME)
print(f"Checkpoint: {checkpoint_path}")

generator, device = load_generator(checkpoint_path)
print(f"Device: {device}")

## 4. Run inference and display result

In [ ]:
result = run_inference_on_paired_image(
    paired_image_path=INPUT_IMAGE,
    generator=generator,
    device=device,
)

fig = make_side_by_side_figure(result, model_name=MODEL_NAME)
plt.show()

## 5. (Optional) Compare all models on the same image

Side-by-side comparison of all five trained models on the same input. This is the same kind of figure included in the project report.

In [ ]:
import numpy as np

# Load each model once and run inference on the same image
outputs_per_model = {}

for name in AVAILABLE_MODELS:
    ckpt = resolve_checkpoint_path(name)
    g, dev = load_generator(ckpt)
    res = run_inference_on_paired_image(
        paired_image_path=INPUT_IMAGE,
        generator=g,
        device=dev,
    )
    outputs_per_model[name] = res
    del g

# Build a grid: label / each model's output / real
any_result = next(iter(outputs_per_model.values()))
model_names = list(outputs_per_model.keys())
num_cols = 2 + len(model_names)

fig, axes = plt.subplots(1, num_cols, figsize=(3 * num_cols, 4))

axes[0].imshow(any_result["label"])
axes[0].set_title("Label map")
axes[0].axis("off")

for col, name in enumerate(model_names, start=1):
    axes[col].imshow(outputs_per_model[name]["generated"])
    axes[col].set_title(name)
    axes[col].axis("off")

axes[-1].imshow(any_result["real"])
axes[-1].set_title("Real (ground truth)")
axes[-1].axis("off")

plt.tight_layout()
plt.show()